# 🕵️ Lab W5-2 — Data Leakage และ Temporal Validation

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 5 — Data Mining I**

Lab นี้ใช้คู่กับสื่อจำลอง **Leakage Hunter** (`/sims/leakage-hunter`)
ตัวเลขที่คุณคำนวณได้ในสมุดเล่มนี้ต้องตรงกับตัวเลขบนหน้าจอสื่อจำลองทุกหลัก

## สิ่งที่จะได้เรียนรู้
1. ตรวจจับ **data leakage** ได้ก่อนที่โมเดลจะขึ้นใช้งานจริง
2. อธิบายว่าเหตุใด **การแบ่งข้อมูลแบบสุ่มจึงปกปิดปัญหา** ที่การแบ่งตามเวลาเปิดเผย
3. วัด **calibration** ไม่ใช่แค่ AUC และอธิบายว่าเหตุใดจึงสำคัญกว่าในงานสินเชื่อ
4. เขียน **checklist ตรวจการรั่ว** ที่นำไปใช้กับโปรเจกต์อื่นได้

## ข้อมูล
`loan_leaky.csv` — คำขอสินเชื่อ 12,000 รายการ ปี 2023–2025

ตารางนี้ถูกดึงจากระบบปฏิบัติการ **ณ วันนี้** จึงมีคอลัมน์ที่บันทึกเหตุการณ์
ซึ่งเกิดขึ้น **หลัง** การอนุมัติปนอยู่ด้วย

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

URL = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
       "master/datasets/week05/loan_leaky.csv")
df = pd.read_csv(URL)
df["status_bad"] = (df.account_status == "ค้างชำระ").astype(int)

print(f"จำนวนคำขอ   : {len(df):,}")
print(f"ช่วงเวลา    : {df.application_date.min()} ถึง {df.application_date.max()}")
print(f"อัตราผิดนัด : {df.defaulted.mean()*100:.2f}%")
print(f"\nคอลัมน์ทั้งหมด:\n  " + "\n  ".join(df.columns))

## ส่วนที่ 1 — ตรวจจับการรั่วโดยยังไม่ต้องฝึกโมเดล

วิธีที่เร็วที่สุดคือดู **AUC ของตัวแปรเดี่ยว** เทียบกับคำตอบ
ตัวแปรใดที่เพียงตัวเดียวก็แยกได้เกือบสมบูรณ์ แทบจะรับประกันได้ว่ารั่ว

In [ ]:
def auc(scores, y) -> float:
    """AUC ด้วยวิธี Mann–Whitney U รองรับค่าที่ซ้ำกันด้วยอันดับเฉลี่ย"""
    r = pd.Series(np.asarray(scores, dtype=float)).rank()
    y = np.asarray(y)
    n_pos = int(y.sum())
    n_neg = len(y) - n_pos
    return (r[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


LEGIT = ["income_at_application", "debt_ratio_at_application",
         "credit_history_months_at_application", "age_at_application",
         "prev_loans_at_application", "loan_amount_at_application"]
SUSPECT = ["collection_calls", "days_since_last_payment", "status_bad"]

### 🧑‍💻 งานที่ 1
คำนวณ AUC ของทุกตัวแปรเดี่ยว แล้วเรียงจากค่าที่ห่างจาก 0.50 มากที่สุด
จากนั้นระบุว่าตัวแปรใดน่าสงสัยว่ารั่ว พร้อมเหตุผลเชิงเวลา (ไม่ใช่เชิงสถิติ)

*เฉลยที่ถูกต้อง: ตัวแปรที่รั่วทั้ง 3 ตัวได้ AUC = 1.0000 พอดี
ส่วนตัวแปรที่ถูกต้องอยู่ในช่วง 0.42–0.63*

In [ ]:
solo = pd.DataFrame([
    {"ตัวแปร": c, "AUC เดี่ยว": auc(df[c], df.defaulted)}
    for c in LEGIT + SUSPECT
])
solo["ห่างจาก 0.50"] = (solo["AUC เดี่ยว"] - 0.5).abs()
solo = solo.sort_values("ห่างจาก 0.50", ascending=False).set_index("ตัวแปร")
print(solo.to_string())

print("""
ตัวแปรที่รั่ว และเหตุผลเชิงเวลา
--------------------------------
collection_calls        จำนวนครั้งที่โทรทวงหนี้ — ต้องมีการค้างชำระเกิดขึ้นก่อน
                        ณ วินาทีที่ต้องตัดสินใจอนุมัติ ค่านี้เป็น 0 เสมอทุกราย

days_since_last_payment ต้องมีการชำระเกิดขึ้นก่อน จึงเป็นข้อมูลหลังอนุมัติทั้งหมด

status_bad              สถานะบัญชี = ค้างชำระ แทบเป็นคำตอบโดยตรง
                        บัญชีที่ค้างชำระคือนิยามของการผิดนัดชำระอยู่แล้ว

สังเกตว่าเหตุผลทั้งสามข้อไม่มีคำว่า 'AUC' อยู่เลย
สถิติเป็นเพียงสิ่งที่ทำให้เราหันไปมอง — สิ่งที่ตัดสินคือคำถามว่า
'ณ วินาทีที่ต้องตัดสินใจ ค่านี้มีอยู่แล้วหรือยัง'
""")

> **หลักการ** ถ้าตัวแปรเดี่ยวให้ AUC เกิน 0.90 ในปัญหาที่ยากโดยธรรมชาติ
> ให้ถือว่า **รั่วไว้ก่อน** จนกว่าจะพิสูจน์ได้ว่าไม่รั่ว
> ไม่ใช่ถือว่าเป็นตัวแปรที่ดีจนกว่าจะพิสูจน์ได้ว่ารั่ว

## ส่วนที่ 2 — ฝึกโมเดลทั้ง 4 กรณี

ใช้ logistic regression แบบ gradient descent ที่เขียนเอง
เพื่อให้ผลตรงกับสื่อจำลองทุกทศนิยม (สื่อจำลองใช้อัลกอริทึมเดียวกันนี้ในเบราว์เซอร์)

In [ ]:
def fit_logistic(X, y, epochs=400, lr=0.5):
    """gradient descent เต็มชุด เริ่มจากน้ำหนักศูนย์ — ผลเหมือนกันทุกครั้ง"""
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        p = 1 / (1 + np.exp(-Xb @ w))
        w -= lr * (Xb.T @ (p - y)) / len(y)
    return w


def predict(X, w):
    return 1 / (1 + np.exp(-(np.c_[np.ones(len(X)), X] @ w)))

### 🧑‍💻 งานที่ 2
เขียนฟังก์ชัน `experiment(cols, split)` ที่

* `split="random"` → สลับลำดับด้วย `random_state=42` แล้วแบ่ง 70/30
* `split="temporal"` → เรียงตาม `application_date` แล้วตัด 70% แรกเป็นชุดฝึก
* มาตรฐานฟีเจอร์ด้วยค่าเฉลี่ยและส่วนเบี่ยงเบนของ **ชุดฝึกเท่านั้น**
* คืน AUC ของชุดฝึกและชุดทดสอบ · อัตราผิดนัดที่ทำนาย · อัตราที่เกิดขึ้นจริง

แล้วรันทั้ง 4 กรณี (มี/ไม่มีตัวแปรรั่ว × สุ่ม/ตามเวลา)

*เฉลยที่ถูกต้อง: ตัวแปรที่ถูกต้อง + แบ่งตามเวลา → AUC ทดสอบ 0.6864
ทำนาย 8.10% แต่เกิดขึ้นจริง 12.03%*

In [ ]:
def experiment(cols, split):
    if split == "random":
        d = df.sample(frac=1, random_state=42)
    else:
        d = df.sort_values("application_date")

    k = int(len(d) * 0.7)
    tr, te = d.iloc[:k], d.iloc[k:]

    # มาตรฐานจากชุดฝึกเท่านั้น — ถ้าใช้ทั้งชุดคือการรั่วอีกแบบหนึ่ง
    mu = tr[cols].astype(float).mean()
    sd = tr[cols].astype(float).std().replace(0, 1)
    Xtr = ((tr[cols].astype(float) - mu) / sd).values
    Xte = ((te[cols].astype(float) - mu) / sd).values

    w = fit_logistic(Xtr, tr.defaulted.values)
    p_te = predict(Xte, w)
    return {
        "AUC ฝึก": auc(predict(Xtr, w), tr.defaulted),
        "AUC ทดสอบ": auc(p_te, te.defaulted),
        "ทำนาย %": p_te.mean() * 100,
        "จริง %": te.defaulted.mean() * 100,
        "ทดสอบตั้งแต่": te.application_date.min(),
        "ถึง": te.application_date.max(),
    }


results = []
for name, cols in [("ถูกต้องตามเวลา", LEGIT), ("มีตัวแปรรั่ว", LEGIT + SUSPECT)]:
    for split in ["random", "temporal"]:
        results.append({"ตัวแปร": name, "การแบ่ง": split, **experiment(cols, split)})

out = pd.DataFrame(results).set_index(["ตัวแปร", "การแบ่ง"])
print(out.to_string())

> **สิ่งที่ต้องสังเกตให้ได้ 3 ข้อ**
>
> 1. **AUC = 1.0000 ทั้งสองแบบเมื่อมีตัวแปรรั่ว** — การแบ่งตามเวลาไม่ได้ช่วยตรวจจับการรั่ว
>    เพราะตัวแปรที่รั่วก็รั่วอยู่ทั้งในชุดฝึกและชุดทดสอบเท่ากัน
> 2. **ไม่มี error ใดเกิดขึ้นเลย** โมเดลรันผ่าน ตัวเลขสวย และผิดทั้งหมด
> 3. AUC ของโมเดลที่ถูกต้องอยู่ราว 0.68–0.69 ซึ่ง **เป็นค่าปกติของปัญหานี้ในอุตสาหกรรม**
>    ไม่ใช่สัญญาณว่าโมเดลไม่ดี

## ส่วนที่ 3 — สิ่งที่การแบ่งตามเวลาเปิดเผย

ถ้าการแบ่งตามเวลาไม่ได้ช่วยตรวจการรั่ว แล้วมันมีไว้ทำไม

### 🧑‍💻 งานที่ 3
เปรียบเทียบ **calibration** (อัตราที่ทำนาย เทียบกับอัตราที่เกิดขึ้นจริง)
ของโมเดลที่ใช้ตัวแปรถูกต้อง ระหว่างการแบ่งสองแบบ

แล้วตอบว่าการแบ่งแบบใดกำลังโกหก และโกหกเรื่องอะไร

In [ ]:
for split in ["random", "temporal"]:
    r = experiment(LEGIT, split)
    gap = r["จริง %"] - r["ทำนาย %"]
    print(f"การแบ่งแบบ {split}")
    print(f"  ชุดทดสอบครอบคลุม : {r['ทดสอบตั้งแต่']} ถึง {r['ถึง']}")
    print(f"  AUC ทดสอบ        : {r['AUC ทดสอบ']:.4f}")
    print(f"  ทำนายผิดนัด      : {r['ทำนาย %']:.2f}%")
    print(f"  เกิดขึ้นจริง      : {r['จริง %']:.2f}%")
    print(f"  ประเมินต่ำกว่าจริง : {gap:.2f} จุด ({gap/r['ทำนาย %']*100:.0f}% ของค่าที่ทำนาย)\n")

print("""คำตอบ
-----
AUC ของทั้งสองแบบแทบเท่ากัน (0.6925 กับ 0.6864) ถ้าดูแค่ AUC จะสรุปว่าไม่ต่างกัน

แต่ calibration ต่างกันอย่างสิ้นเชิง
  แบ่งแบบสุ่ม   ทำนาย 9.37% เกิดจริง 9.53%  → ห่างกันไม่ถึงครึ่งจุด ดูสอบเทียบดีมาก
  แบ่งตามเวลา   ทำนาย 8.10% เกิดจริง 12.03% → ประเมินต่ำกว่าจริง 3.92 จุด หรือ 48%

การแบ่งแบบสุ่มกำลังโกหกเรื่อง 'ความพร้อมใช้งานจริง'
เพราะชุดทดสอบของมันมีคำขอจากทุกช่วงเวลาปนกัน รวมถึงช่วงที่อยู่ในชุดฝึกด้วย
โมเดลจึงได้เห็นสภาพตลาดของช่วงเวลาที่มันต้องไปทำนายมาแล้ว

ในงานสินเชื่อ calibration สำคัญกว่า AUC เพราะธนาคารต้องใช้ค่าความน่าจะเป็น
ไปตั้งสำรองหนี้สูญและตั้งราคาดอกเบี้ย โมเดลที่จัดอันดับความเสี่ยงได้ถูก (AUC ดี)
แต่บอกระดับความเสี่ยงต่ำกว่าจริง 50% จะทำให้ธนาคารตั้งสำรองไม่พออย่างเป็นระบบ
""")

## ส่วนที่ 4 — ต้นตอคือ population drift

### 🧑‍💻 งานที่ 4
แสดงอัตราผิดนัดชำระรายครึ่งปี แล้วอธิบายว่าเหตุใดโมเดลที่ฝึกจากอดีต
จึงประเมินความเสี่ยงของอนาคตต่ำกว่าจริง **เสมอ** ไม่ใช่บางครั้ง

*เฉลยที่ถูกต้อง: อัตราผิดนัดเพิ่มจาก 6.58% เป็น 12.30% ตลอด 3 ปี*

In [ ]:
df["half"] = (df.application_date.str[:4] + "-H"
              + np.where(df.application_date.str[5:7].astype(int) <= 6, "1", "2"))

drift = df.groupby("half").defaulted.agg(["size", "mean"])
drift.columns = ["จำนวนคำขอ", "อัตราผิดนัด"]
drift["อัตราผิดนัด %"] = drift["อัตราผิดนัด"] * 100
print(drift[["จำนวนคำขอ", "อัตราผิดนัด %"]].to_string())

first, last = drift["อัตราผิดนัด"].iloc[0], drift["อัตราผิดนัด"].iloc[-2]
print(f"\nเพิ่มขึ้นจาก {first*100:.2f}% เป็น {last*100:.2f}% "
      f"({(last/first-1)*100:.0f}% เชิงสัมพัทธ์)")

print("""
เหตุใดจึงประเมินต่ำกว่าจริง 'เสมอ'
-----------------------------------
โมเดลเรียนรู้ค่าคงที่ (intercept) จากอัตราผิดนัดเฉลี่ยของชุดฝึก
เมื่อชุดฝึกคือช่วงต้นที่อัตราอยู่ราว 8.2% และชุดทดสอบคือช่วงปลายที่อัตราขึ้นเป็น 12.0%
โมเดลย่อมทำนายรอบ ๆ 8% ไม่ว่าจะจัดอันดับรายบุคคลได้ดีเพียงใด

ตราบใดที่แนวโน้มยังเป็นขาขึ้น ทิศทางของความคลาดเคลื่อนจะเป็นทางเดียวเสมอ
ไม่ใช่ความผิดพลาดแบบสุ่มที่จะหักล้างกันเอง — จึงสะสมเป็นความเสียหายจริง

ทางแก้ไม่ใช่ 'หาโมเดลที่ดีกว่า' แต่คือ
  1. ฝึกใหม่ตามรอบด้วยข้อมูลล่าสุด
  2. เฝ้าดู calibration ในการใช้งานจริง ไม่ใช่แค่ AUC ตอนพัฒนา
  3. ปรับค่าคงที่ (recalibration) เมื่อพบว่าคลาดเคลื่อนเป็นระบบ
""")

## ส่วนที่ 5 — เปรียบเทียบกับ scikit-learn

### 🧑‍💻 งานที่ 5
ทำซ้ำกรณี "ตัวแปรถูกต้อง + แบ่งตามเวลา" ด้วย
`sklearn.linear_model.LogisticRegression` แล้วเทียบ AUC กับที่คำนวณเอง

จากนั้นตอบว่า ถ้าใช้ `cross_val_score` แบบ 5-fold ธรรมดา
จะตรวจจับปัญหาที่พบใน Lab นี้ได้หรือไม่ เพราะเหตุใด

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = df.sort_values("application_date")
k = int(len(d) * 0.7)
tr, te = d.iloc[:k], d.iloc[k:]

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(tr[LEGIT], tr.defaulted)
sk_auc = roc_auc_score(te.defaulted, model.predict_proba(te[LEGIT])[:, 1])

print(f"AUC จาก sklearn      : {sk_auc:.4f}")
print(f"AUC จากโค้ดที่เขียนเอง : {experiment(LEGIT,'temporal')['AUC ทดสอบ']:.4f}")
print("(ต่างกันเล็กน้อยเพราะ sklearn ใช้ตัวหาโซลูชันที่ลู่เข้าสมบูรณ์กว่า gradient descent 400 รอบ)")

cv_random = cross_val_score(model, df[LEGIT], df.defaulted, cv=5, scoring="roc_auc")
cv_time = cross_val_score(model, d[LEGIT], d.defaulted,
                          cv=TimeSeriesSplit(n_splits=5), scoring="roc_auc")
print(f"\n5-fold ธรรมดา (สุ่ม)     AUC = {cv_random.mean():.4f} ± {cv_random.std():.4f}")
print(f"TimeSeriesSplit (ตามเวลา) AUC = {cv_time.mean():.4f} ± {cv_time.std():.4f}")

print("""
คำตอบ: cross_val_score แบบ 5-fold ธรรมดา 'ตรวจไม่พบ' ทั้งสองปัญหา
------------------------------------------------------------------
1. ตรวจไม่พบการรั่ว — เพราะทุก fold มีตัวแปรที่รั่วเหมือนกันหมด
   ทุก fold จึงได้ AUC ใกล้ 1.00 พร้อมกัน และค่าเบี่ยงเบนมาตรฐานก็ต่ำ
   ซึ่งดู 'เสถียร' ทั้งที่ผิดทั้งกระบิ

2. ตรวจไม่พบ drift — เพราะ KFold สุ่มแบ่งโดยไม่สนใจเวลา
   ทุก fold จึงมีข้อมูลจากทุกช่วงเวลาปนกัน เหมือนการแบ่งแบบสุ่มในงานที่ 3

การตรวจสอบไขว้ที่ 'จำนวน fold มากขึ้น' ไม่ได้แปลว่า 'เชื่อถือได้มากขึ้น'
ถ้าโครงสร้างของการแบ่งไม่ตรงกับวิธีที่โมเดลจะถูกใช้จริง
สำหรับข้อมูลที่มีมิติเวลา ต้องใช้ TimeSeriesSplit เสมอ
""")

## ส่วนที่ 6 — Checklist ที่นำไปใช้ต่อได้

### 🧑‍💻 งานที่ 6 (เขียนเป็นข้อความ)

1. เขียน checklist ตรวจการรั่วอย่างน้อย 6 ข้อ ที่ทีมของคุณจะใช้กับ**ทุกโปรเจกต์**
   แต่ละข้อต้องเป็นคำถามที่ตอบได้ว่าใช่หรือไม่ ไม่ใช่หลักการลอย ๆ
2. ยกตัวอย่างการรั่วอีก 2 แบบที่ไม่ปรากฏใน Lab นี้ พร้อมบริบทธุรกิจ
3. ถ้าคุณเป็นผู้ตรวจสอบโมเดลของทีมอื่น และเขาส่งโมเดลที่ AUC 0.94 มาให้
   คุณจะขอดูอะไรบ้างก่อนอนุมัติ

In [ ]:
CHECKLIST = """
Checklist ตรวจ Data Leakage — ใช้ก่อนส่งมอบโมเดลทุกตัว
========================================================
[ ] 1. ทุกคอลัมน์ตอบได้หรือไม่ว่า 'ณ วินาทีที่ต้องตัดสินใจ ค่านี้มีอยู่แล้ว'
       ถ้าตอบไม่ได้แม้แต่คอลัมน์เดียว ให้ตัดออกก่อน แล้วค่อยพิสูจน์ทีหลัง

[ ] 2. มีตัวแปรเดี่ยวใดให้ AUC เกิน 0.90 หรือไม่
       ถ้ามี ต้องมีเอกสารอธิบายว่าเหตุใดจึงไม่ใช่การรั่ว

[ ] 3. AUC ของโมเดลสูงกว่าค่ามาตรฐานของปัญหาประเภทนี้ในอุตสาหกรรมหรือไม่
       ถ้าใช่ ให้ถือว่ามีบั๊กจนกว่าจะพิสูจน์เป็นอย่างอื่น

[ ] 4. ข้อมูลมีมิติเวลาหรือไม่ ถ้ามี ใช้การแบ่งตามเวลาแล้วหรือยัง

[ ] 5. ค่ามาตรฐาน (mean/std) ตัวเติมค่าว่าง และ encoder ทั้งหมด
       ถูก fit จาก 'ชุดฝึกเท่านั้น' หรือ fit จากข้อมูลทั้งชุด

[ ] 6. รายงาน calibration แล้วหรือยัง ไม่ใช่แค่ AUC
       อัตราที่ทำนายเฉลี่ย ตรงกับอัตราที่เกิดขึ้นจริงในชุดทดสอบหรือไม่

[ ] 7. มีคอลัมน์ใดที่ชื่อบ่งบอกอนาคตหรือไม่ (status, final_, closed_, total_, days_since_)

[ ] 8. ถ้ารันโมเดลนี้ย้อนหลังกับข้อมูล ณ วันที่ตัดสินใจจริง จะมีคอลัมน์ครบหรือไม่
       ข้อนี้ทดสอบได้จริงและตรวจจับได้เกือบทุกกรณี
"""
print(CHECKLIST)

print("""ตัวอย่างคำตอบข้อ 2 — การรั่วอีก 2 แบบ
--------------------------------------
1. Target leakage ผ่านการเติมค่าว่าง
   ทำนายว่าลูกค้าจะซื้อประกันหรือไม่ แล้วเติมค่าว่างของคอลัมน์ 'วงเงินคุ้มครอง'
   ด้วยค่าเฉลี่ยของทั้งชุดข้อมูล — ลูกค้าที่ไม่ซื้อจะมีค่าว่าง ค่าที่เติมจึงกลายเป็นสัญญาณ
   ว่าแถวนี้เป็นกลุ่มใด ทั้งที่ตอนใช้งานจริงยังไม่รู้คำตอบ

2. Group leakage ในข้อมูลที่มีหลายแถวต่อหนึ่งหน่วย
   ทำนายการลาออกของพนักงาน โดยแต่ละคนมีหลายแถว (รายเดือน)
   ถ้าสุ่มแบ่งระดับแถว พนักงานคนเดียวกันจะอยู่ทั้งในชุดฝึกและชุดทดสอบ
   โมเดลจึง 'จำคนได้' แทนที่จะเรียนรู้รูปแบบ — ต้องแบ่งที่ระดับพนักงาน (GroupKFold)
""")

---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — ตรวจ AUC ตัวแปรเดี่ยวและให้เหตุผลเชิงเวลา | 3 |
| งานที่ 2 — ทดลองครบ 4 กรณีและได้ตัวเลขตรงเฉลย | 4 |
| งานที่ 3 — วิเคราะห์ calibration และระบุว่าการแบ่งใดโกหก | 4 |
| งานที่ 4 — แสดง drift และอธิบายว่าเหตุใดจึงเอนไปทางเดียว | 3 |
| งานที่ 5 — เทียบ sklearn และอธิบายข้อจำกัดของ 5-fold ธรรมดา | 3 |
| งานที่ 6 — checklist และตัวอย่างการรั่วเพิ่มเติม | 3 |
| **รวม** | **20** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/leakage-hunter`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง